In [19]:
import torch
from torch import nn
import timm
import json
from HybridNAS import HybridNAS
from CompressedViT import CompressedViT
from Pruning.PruneUtils import compute_grads
from torchvision import transforms
from torchvision.datasets import ImageFolder

device = "cuda" if torch.cuda.is_available() else "cpu"

In [20]:
with open("best_architecture.json", "r") as f:
    search_dict = json.load(f)

num_classes = 204
model = timm.create_model("vit_small_patch16_224", pretrained=False, num_classes=num_classes)
checkpoint = torch.load("D:\\Tesi\\FirstFineTuning\\best_model.pth")
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [21]:
pruned_model = HybridNAS.apply_pruning(None, search_dict, model)
comp_model = CompressedViT(search_dict, pruned_model)

In [22]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())


orig_params = count_params(pruned_model)
new_params = count_params(comp_model)
print(f"Originale: {orig_params / 1e6:.2f}M")
print(f"Compresso: {new_params / 1e6:.2f}M")
print(f"Riduzione: {100 * (orig_params - new_params) / orig_params:.2f}%")

Originale: 21.74M
Compresso: 18.54M
Riduzione: 14.75%


In [27]:
def verify_layernorm_crash_test(masked_model, compressed_model, search_dict, device):
    print(f"\n{'='*20} AUTOPSIA LAYERNORM (BLOCCO 0) {'='*20}")

    masked_model.eval()
    compressed_model.eval()

    # 1. Creiamo lo stesso input
    x = torch.randn(1, 3, 224, 224).to(device)

    # Indici attivi (per filtrare il modello mascherato)
    # Ci serve sapere quali canali "sopravvivono" per confrontare mele con mele
    new_emb_dims = [i for i in range(384) if i not in search_dict["embed_pruned_dims"]]

    # ---------------------------------------------------------
    # STEP 1: INPUT ALLA NORM (Patch Embed + Pos Embed)
    # ---------------------------------------------------------
    print("\n--- STEP 1: Input alla LayerNorm ---")

    # Calcolo Masked
    x_m = masked_model.patch_embed(x)
    x_m = masked_model._pos_embed(x_m) # Aggiunge pos_embed e cls_token

    # Calcolo Compressed
    x_c = compressed_model.patch_embed(x)
    x_c = compressed_model.pos_enc(x_c)

    # Filtriamo il Masked per confrontarlo
    x_m_filtered = x_m[:, :, new_emb_dims]

    diff_input = (x_m_filtered - x_c).abs().max().item()
    print(f"Differenza Input: {diff_input:.8f}")

    if diff_input > 1e-4:
        print("❌ FERMI TUTTI: L'input è già diverso! Il problema è prima.")
        return

    print("✅ L'input è IDENTICO. I pesi dell'embedding sono corretti.")

    # ---------------------------------------------------------
    # STEP 2: DIMOSTRAZIONE MATEMATICA (Media e Varianza)
    # ---------------------------------------------------------
    print("\n--- STEP 2: Analisi Statistica Interna ---")

    # Prendiamo il primo token della prima immagine per analizzarlo
    token_m = x_m[0, 0, :]  # Vettore lungo 384 (con zeri)
    token_c = x_c[0, 0, :]  # Vettore lungo ~330 (senza zeri)

    # Calcoliamo manualmente quello che fa la LayerNorm
    mean_m = token_m.mean()
    var_m = token_m.var(unbiased=False)

    mean_c = token_c.mean()
    var_c = token_c.var(unbiased=False)

    print(f"Media Originale (con zeri):    {mean_m.item():.6f}")
    print(f"Media Compressa (senza zeri):  {mean_c.item():.6f}")
    print(f" -> Delta Media:               {abs(mean_m.item() - mean_c.item()):.6f}")

    print(f"Varianza Originale:            {var_m.item():.6f}")
    print(f"Varianza Compressa:            {var_c.item():.6f}")

    # ---------------------------------------------------------
    # STEP 3: OUTPUT DELLA NORM
    # ---------------------------------------------------------
    print("\n--- STEP 3: Uscita dalla LayerNorm ---")

    # Applichiamo la Norm1 del Blocco 0
    # Nota: masked usa .norm1, compressed usa .layerNorm1 (nome diverso nella tua classe)
    out_m = masked_model.blocks[0].norm1(x_m)
    out_c = compressed_model.blocks[0].layerNorm1(x_c)

    # Filtriamo l'output mascherato
    out_m_filtered = out_m[:, :, new_emb_dims]

    diff_output = (out_m_filtered - out_c).abs().max().item()
    print(f"Differenza Output: {diff_output:.6f}")

    if diff_output > 0.1:
        print("\n💥 ECCO IL COLPEVOLE! La differenza è esplosa nella Norm.")
        print("La stessa matematica applicata a statistiche diverse produce numeri diversi.")

# Esegui il test
verify_layernorm_crash_test(pruned_model, comp_model, search_dict, device)


==================== AUTOPSIA LAYERNORM (BLOCCO 0) ====================

--- STEP 1: Input alla LayerNorm ---
Differenza Input: 0.00000000
✅ L'input è IDENTICO. I pesi dell'embedding sono corretti.

--- STEP 2: Analisi Statistica Interna ---
Media Originale (con zeri):    -0.067214
Media Compressa (senza zeri):  -0.077977
 -> Delta Media:               0.010762
Varianza Originale:            1.328856
Varianza Compressa:            1.540795

--- STEP 3: Uscita dalla LayerNorm ---
Differenza Output: 0.347061

💥 ECCO IL COLPEVOLE! La differenza è esplosa nella Norm.
La stessa matematica applicata a statistiche diverse produce numeri diversi.


In [25]:
data_config = timm.data.resolve_model_data_config(model)
imagenet_mean, imagenet_std = data_config["mean"], data_config["std"]

search_transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize(mean=imagenet_mean, std=imagenet_std)])

path = "D:\\Tesi\\Sets\\Set1\\search"
batch_size = 128

search_set = ImageFolder(root=path, transform=search_transform)
search_loader = torch.utils.data.DataLoader(search_set, batch_size=batch_size, shuffle=False, num_workers=1)
pruned_model = pruned_model.to(device)
pruned_model.eval()
_, final_accuracy = compute_grads(pruned_model, nn.CrossEntropyLoss(), device, search_loader)
print(final_accuracy)

0.5352941176470588


In [28]:
print(comp_model)

CompressedViT(
  (patch_embed): PatchEmbed(
    (patch_embed): Conv2d(3, 331, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_enc): PositionalEncoding()
  (blocks): Sequential(
    (0): EncoderBlock(
      (mhsa): MultiHeadSelfAttention(
        (qkv): Linear(in_features=331, out_features=1152, bias=True)
        (proj): Linear(in_features=384, out_features=331, bias=True)
      )
      (layerNorm1): LayerNorm((331,), eps=1e-06, elementwise_affine=True)
      (layerNorm2): LayerNorm((331,), eps=1e-06, elementwise_affine=True)
      (fc1): Linear(in_features=331, out_features=1536, bias=True)
      (fc2): Linear(in_features=1536, out_features=331, bias=True)
      (mlp): Sequential(
        (0): Linear(in_features=331, out_features=1536, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1536, out_features=331, bias=True)
      )
    )
    (1): EncoderBlock(
      (mhsa): MultiHeadSelfAttention(
        (qkv): Linear(in_features=331, out_features=1152, bi